# Agent function calls

A `Button` can call a function by name. The browser runs the functions it knows, such as `openUrl`. For any other name, it sends a `callAgentFunction` message to the agent. Here the kernel plays the agent role.

The handler below answers `getForecast` calls with an `agentFunctionResponse`, which carries the same `functionCallId`. It also writes the forecast into the data model.

In [ ]:
import sys
from importlib.util import find_spec

# Pyodide and xeus-python both report sys.platform "emscripten"; only Pyodide ships the pyodide package.
if sys.platform != "emscripten":
    KERNEL = "cpython"
else:
    try:
        has_pyodide = find_spec("pyodide") is not None
    except ValueError:  # xeus-python's pyodide stub has __spec__ set to None
        has_pyodide = False
    KERNEL = "pyodide" if has_pyodide else "xpython"

if KERNEL == "pyodide" and find_spec("a2ui_jupyter") is None:
    import piplite

    await piplite.install(["rpds-py", "a2ui-jupyter"])

from a2ui_jupyter import A2UISurface

BASIC_CATALOG_ID = "https://a2ui.org/specification/v1_0/catalogs/basic/catalog.json"


def new_surface(surface_id, *, send_data_model=False, **widget_kwargs):
    surface = A2UISurface(**widget_kwargs)
    create = {"surfaceId": surface_id, "catalogId": BASIC_CATALOG_ID}
    if send_data_model:
        create["sendDataModel"] = True
    surface.send({"version": "1.0", "createSurface": create})
    return surface

In [ ]:
SURFACE_ID = "agent-function-call"
FORECASTS = {
    "Seattle": "Light rain, 12 C",
    "Lisbon": "Sunny, 21 C",
}

surface = new_surface(SURFACE_ID)


def answer_agent_calls(message):
    payload = message.model_dump(by_alias=True, exclude_unset=True)
    if "callAgentFunction" not in payload:
        return
    call = payload["callAgentFunction"]
    function = call["callFunction"]
    if function["@call"] != "getForecast":
        return
    city = function["args"]["city"]
    forecast = FORECASTS.get(city, "Unknown city")
    surface.send(
        [
            {
                "version": "1.0",
                "agentFunctionResponse": {
                    "functionCallId": call["functionCallId"],
                    "value": forecast,
                },
            },
            {
                "version": "1.0",
                "updateDataModel": {
                    "surfaceId": SURFACE_ID,
                    "path": "/forecast",
                    "value": f"{city}: {forecast}",
                },
            },
        ]
    )


unsubscribe = surface.on_message(answer_agent_calls)

surface.send(
    [
        {
            "version": "1.0",
            "updateDataModel": {
                "surfaceId": SURFACE_ID,
                "value": {"forecast": "No forecast yet"},
            },
        },
        {
            "version": "1.0",
            "updateComponents": {
                "surfaceId": SURFACE_ID,
                "components": [
                    {
                        "id": "root",
                        "component": "Column",
                        "children": ["title", "seattle_button", "lisbon_button", "forecast"],
                        "align": "center",
                    },
                    {"id": "title", "component": "Text", "text": "## Forecast lookup"},
                    {
                        "id": "seattle_button",
                        "component": "Button",
                        "child": "seattle_label",
                        "action": {
                            "functionCall": {"@call": "getForecast", "args": {"city": "Seattle"}}
                        },
                    },
                    {"id": "seattle_label", "component": "Text", "text": "Seattle"},
                    {
                        "id": "lisbon_button",
                        "component": "Button",
                        "child": "lisbon_label",
                        "action": {
                            "functionCall": {"@call": "getForecast", "args": {"city": "Lisbon"}}
                        },
                    },
                    {"id": "lisbon_label", "component": "Text", "text": "Lisbon"},
                    {
                        "id": "forecast",
                        "component": "Text",
                        "text": {"@path": "/forecast"},
                        "variant": "caption",
                    },
                ],
            },
        },
    ]
)
surface

Click **Seattle** or **Lisbon**. The forecast line updates after the kernel answers the call. The handler stays registered until the cell below runs.

In [ ]:
unsubscribe()
print("Forecast handler removed.")